# Μάθημα 03 - Πρότυπα Σχεδιασμού Ενεργών Παραγόντων

Σε αυτό το μάθημα, εξερευνούμε τρία θεμελιώδη πρότυπα σχεδιασμού για την κατασκευή αποτελεσματικών πρακτόρων AI:

1. **Σαφείς Οδηγίες για τον Πράκτορα** — Δημιουργία ακριβών, ρόλο-οριστικών ερωτημάτων που καθοδηγούν τη συμπεριφορά του πράκτορα
2. **Δομημένη Έξοδος με Μοντέλα Pydantic** — Εξασφάλιση ότι οι πράκτορες επιστρέφουν προβλέψιμα, ελεγμένα δεδομένα
3. **Πράκτορες Μονής Ευθύνης** — Σχεδιασμός εστιασμένων πρακτόρων που κάνουν καλά το ένα πράγμα που αναλαμβάνουν

Θα εφαρμόσουμε κάθε πρότυπο σε ένα σενάριο **συστήματος σύστασης προορισμών ταξιδιού**, οικοδομώντας σταδιακά ένα σύστημα που μπορεί να προτείνει προορισμούς, να ελέγχει διαθεσιμότητα και να χειρίζεται τη λογιστική.


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Πρότυπο 1: Σαφείς Οδηγίες για τον Πράκτορα

Το πιο αποτελεσματικό πρότυπο είναι και το απλούστερο: η γραφή σαφών, λεπτομερέστατων οδηγιών για τον πράκτορά σας.

Καλές οδηγίες ορίζουν:
- **Ποιος** είναι ο πράκτορας (προσωπικότητα και τόνος)
- **Τι** πρέπει να κάνει (ευθύνες βήμα προς βήμα)
- **Πώς** πρέπει να συμπεριφέρεται (περιορισμοί και στυλ)

Παρακάτω, δημιουργούμε έναν ταξιδιωτικό πράκτορα concierge με ρητές οδηγίες που διαμορφώνουν κάθε απάντηση που παράγει.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Πρότυπο 2: Δομημένη έξοδος με Pydantic μοντέλα

Το ελεύθερο κείμενο είναι χρήσιμο για συνομιλίες, αλλά τα συστήματα που ακολουθούν χρειάζονται δομημένα δεδομένα.
Συνδυάζοντας **Pydantic μοντέλα** με μια **λειτουργία εργαλείου**, μπορούμε να:

- Ορίσουμε ένα ακριβές σχήμα για την έξοδο του πράκτορα
- Επικυρώσουμε αυτόματα τις απαντήσεις
- Ενσωματώσουμε αποτελέσματα του πράκτορα στη λογική εφαρμογής αξιόπιστα

Το κλειδί για την επιβολή είναι η παράδοση του `response_format` όταν τρέχουμε τον πράκτορα. Αυτό αναγκάζει το
μοντέλο να επιστρέψει ένα επικυρωμένο αντικείμενο `TravelRecommendations` (διαθέσιμο στο `response.value`)
αντί για ελεύθερο κείμενο. Το εργαλείο `get_destination_details` επιστρέφει επίσης έναν τυποποιημένο
`DestinationRecommendation`, έτσι τα δεδομένα παραμένουν δομημένα από την αρχή ως το τέλος.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Μοτίβο 3: Πράκτορες Μοναδικής Ευθύνης

Οι πολύπλοκες εργασίες ωφελούνται από τη διαίρεση της δουλειάς σε πολλούς εστιασμένους πράκτορες, καθένας με μοναδική ευθύνη:

- Έναν **Ειδικό Προορισμού** που γνωρίζει για μέρη και διαθεσιμότητα
- Έναν **Σχεδιαστή Logistics** που χειρίζεται πτήσεις, ξενοδοχεία και δρομολόγια

Αυτό αντανακλά την αρχή της μηχανικής λογισμικού της *διαχωρισμού των ανησυχιών* — κάθε πράκτορας είναι πιο εύκολος να δοκιμαστεί, να συντηρηθεί και να βελτιωθεί ανεξάρτητα.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Περίληψη

Σε αυτό το μάθημα εφαρμόσαμε τρία πρότυπα σχεδίασης πράκτορα σε ένα σενάριο συστήματος προτάσεων ταξιδιών:

| Πρότυπο | Κεντρική Ιδέα | Όφελος |
|---|---|---|
| **Σαφείς Οδηγίες** | Ορισμός προσωπικότητας, ευθυνών και περιορισμών από την αρχή | Συνεπής, σύμφωνα με το προφίλ συμπεριφορά πράκτορα |
| **Δομημένη Έξοδος** | Χρήση μοντέλων Pydantic ως μορφή απάντησης | Ελεγμένα, μηχανικά αναγνώσιμα αποτελέσματα |
| **Μοναδική Ευθύνη** | Ανάθεση μίας συγκεκριμένης εργασίας σε κάθε πράκτορα | Ευκολότερος έλεγχος, συντήρηση και σύνθεση |

Αυτά τα πρότυπα συνδυάζονται φυσικά — μπορείτε να συνδυάσετε σαφείς οδηγίες με δομημένη έξοδο εντός ενός πράκτορα μοναδικής ευθύνης για να δημιουργήσετε ανθεκτικά, έτοιμα για παραγωγή συστήματα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
